In [37]:
# API-key setup — DO NOT hard-code your key in this cell.

import os

# --- Local (with a .env file) ---
from dotenv import load_dotenv
load_dotenv()
API_KEY = os.environ["GROQ_API_KEY"]

# --- Google Colab (Secrets panel) ---
# from google.colab import userdata
# API_KEY = userdata.get("GROQ_API_KEY")

# TODO: set API_KEY using ONE of the methods above.

# OpenAI-compatible client (works for Groq and OpenAI; Gemini users see their docs):
from openai import OpenAI

client = OpenAI(
    api_key=API_KEY,
    base_url="https://api.groq.com/openai/v1",   # remove this line if using OpenAI itself
)
MODEL = "llama-3.3-70b-versatile"                # or your provider's model name

print("Client ready.")


Client ready.


Section 1 — Talking to an LLM Programmatically

In [38]:
# Part 1.1 — Your first API call

# TODO: Write a helper function you will reuse for the WHOLE lab:

def ask_llm(user_prompt, system_prompt="You are a helpful assistant.",
             temperature=0.7, max_tokens=500):
     response = client.chat.completions.create(
         model=MODEL,
         messages=[
             {"role": "system", "content": system_prompt},
             {"role": "user",   "content": user_prompt},
         ],
         temperature=temperature,
         max_tokens=max_tokens,
     )
     return response.choices[0].message.content, response.usage


answer, usage = ask_llm("Who was the first man to be created?")
print(answer)
print(usage)



According to various religious and mythological traditions, the first man to be created is a topic of debate. Here are a few examples:

1. **Adam (Judaism, Christianity, and Islam)**: In the biblical account, Adam is considered the first human created by God. He is said to have been formed from the earth and given life by God's breath (Genesis 2:7).
2. **Ask and Embla (Norse mythology)**: In Norse mythology, the first humans were Ask and Embla, created by the god Odin from two pieces of driftwood.
3. **Manu (Hindu mythology)**: In Hindu mythology, Manu is considered the first human, created by the god Brahma. He is said to have survived a great flood and become the ancestor of all humans.
4. **Pangu (Chinese mythology)**: In Chinese mythology, Pangu is considered the first human, emerging from a primordial egg and separating the sky and the earth.
5. **Enkidu (Sumerian mythology)**: In Sumerian mythology, Enkidu is created by the goddess Aruru to be a companion for the king Gilgamesh.


Student Reasoning — Anatomy of a call 1. What is the difference between the system and user roles? Give an example of something that belongs in each. 2. What is a token, roughly? Why do API providers bill per token rather than per request?

In [39]:
# Part 1.2 — Temperature: the randomness dial

# TODO: Ask the SAME question 5 times at temperature=0.0 and 5 times at temperature=1.2.
#   A good test question: "Suggest a name for a savings product for market traders in Accra."

# TODO: Print all 10 answers, grouped by temperature.
question = "The names of the top games in 2026"

print("Temperature = 0.0")
for i in range(5):
    answer, usage = ask_llm(question, temperature=0.0)
    print(f"Answer {i+1}: {answer}\n")

print("Temperature = 1.2")
for i in range(5):
    answer, usage = ask_llm(question, temperature=1.2)
    print(f"Answer {i+1}: {answer}\n")

Temperature = 0.0
Answer 1: Since my knowledge cutoff is December 2023, I don't have real-time information on the top games of 2026. However, I can give you an idea of the popular games that were trending in 2023, and some upcoming games that were announced for release in 2024 and beyond. Keep in mind that the gaming landscape is constantly evolving, and new games may have been released since my knowledge cutoff.

That being said, here are some popular games from 2023 and upcoming games that were announced for release in 2024 and beyond:

Popular games from 2023:

1. **Elden Ring** (Action RPG)
2. **God of War Ragnarök** (Action-Adventure)
3. **The Last of Us Part I** (Action-Adventure)
4. **Call of Duty: Modern Warfare II** (First-Person Shooter)
5. **Overwatch 2** (Team-Based Shooter)

Upcoming games announced for release in 2024 and beyond:

1. **Hogwarts Legacy** (Action-Adventure, set in the Harry Potter universe)
2. **Starfield** (Sci-Fi RPG, developed by Bethesda)
3. **The Legen

Student Reasoning — Temperature What did you observe at each temperature? For the loan decision-support system you are about to build, which temperature regime is appropriate, and why?

Section 2 — The Dataset: Loan Application Letters

In [40]:
LETTERS = {
"L001": """Dear Sir/Madam,
My name is Akosua Mensah and I have been selling provisions at Makola Market for 12 years.
I am applying for a loan of GHS 8,000 to buy a deep freezer and expand into frozen foods.
My current stall makes about GHS 900 profit each month. I have saved GHS 2,500 with your
susu scheme over the past two years and I have never missed a contribution. I can repay
GHS 450 monthly over 20 months. My sister, a teacher, will stand as my guarantor.
Thank you for considering my application.""",

"L002": """Hello,
I am Kwame Boateng, a commercial driver in Kumasi. I need GHS 25,000 urgently to repair my
trotro engine and settle some personal debts. Business has been slow but it will surely
pick up after the festive season. I can pay back whenever the money comes. I do not have
collateral at the moment but God willing everything will be fine. Please help me quickly.""",

"L003": """Dear Loan Committee,
I am Efua Darko, owner of Darko Fashions, a registered dressmaking business in Takoradi
(registration no. BN-2019-4482). I employ three apprentices. I request GHS 15,000 to
purchase two industrial sewing machines and fabric stock ahead of the Christmas season.
Last year my December revenue alone was GHS 22,000; monthly profit averages GHS 2,800.
I hold a fixed deposit of GHS 5,000 with GCB which I can pledge. Proposed repayment:
GHS 1,100 monthly for 15 months. Attached are my sales records for the past 18 months.""",

"L004": """Good day,
My name is Yaw Owusu. I want a loan for my poultry farm at Nsawam. The amount is GHS 12,000
for feed and 500 new layers. I started the farm last year. Sometimes I make good money,
around GHS 1,500 in a good month, but bird flu affected us in March and I lost many birds.
I am rebuilding now. I can repay in 18 months. My uncle has agreed to guarantee the loan
with his taxi.""",

"L005": """Dear Manager,
I am writing on behalf of the Adenta Women's Weaving Cooperative (14 members). We seek
GHS 30,000 to buy a bulk order of yarn directly from the factory, cutting out middlemen and
raising our margins from 15% to about 35%. The cooperative has operated for 6 years and
holds GHS 9,000 in our group account. We propose repayment of GHS 2,000 monthly over
16 months, backed by our group savings and joint liability agreement.""",

"L006": """Hi,
This is Kofi. I saw your advert. I want GHS 50,000 to start a car washing business, a
provision shop, and also import phones from Dubai. I am 22 and full of energy. I have not
started any of these yet but my friends say I am very business minded. I will pay back in
one year when the businesses are booming. No collateral but I am trustworthy.""",
}

# Gold-standard labels for three letters (for Section 4 evaluation):
GOLD = {
  "L001": {"applicant_name": "Akosua Mensah", "amount_ghs": 8000,  "purpose": "buy deep freezer / expand into frozen foods",
           "monthly_profit_ghs": 900,  "has_collateral_or_guarantor": True,  "repayment_months": 20},
  "L003": {"applicant_name": "Efua Darko",    "amount_ghs": 15000, "purpose": "industrial sewing machines and fabric stock",
           "monthly_profit_ghs": 2800, "has_collateral_or_guarantor": True,  "repayment_months": 15},
  "L006": {"applicant_name": "Kofi",          "amount_ghs": 50000, "purpose": "car wash, provision shop, phone imports",
           "monthly_profit_ghs": None, "has_collateral_or_guarantor": False, "repayment_months": 12},
}

print(f"{len(LETTERS)} letters loaded.")



6 letters loaded.


Section 3 — Prompt Engineering for the Decision Support System

In [41]:
# Part 3.1 — Component 1: Summarization

# TODO: Write SUMMARY_PROMPT_V1 — your first, naive attempt (e.g. just "Summarize this:").
#   Run it on L002 and L006. Read the output critically.
SUMMARY_PROMPT_V1 = "Summarize this:"

for letter_id in ["L002", "L006"]:
    prompt = f"{SUMMARY_PROMPT_V1}\n\n{LETTERS[letter_id]}"
    answer, usage, = ask_llm(prompt)
    print(f"V1 on {letter_id}")
    print(answer)
    print()



# TODO: Now write SUMMARY_PROMPT_V2 as a proper template with:
SUMMARY_PROMPT_V2_SYSTEM = (
    "You are an assistant to a microfinance loan officer."
    "Summarize loan applications factually and neutrally."
    "No invention of details which have not been explicitly stated in the letter."
    "Respond in exactly 3-4 sentences."
)

for letter_id in ["L002", "L006"]:
    user_prompt   = f"Summarize this loan application: \n\n{LETTERS[letter_id]}"
    answer, usage = ask_llm(user_prompt, system_prompt=SUMMARY_PROMPT_V2_SYSTEM, temperature=0)
    print(f"V2 on {letter_id}")
    print(answer)
    print()

# TODO: Compare V1 vs V2 outputs side by side. Keep both prompt versions in this notebook.


V1 on L002
Kwame Boateng, a commercial driver from Kumasi, is requesting a loan of GHS 25,000 to repair his vehicle's engine and pay off personal debts. He's experiencing a slow business period but expects it to improve after the festive season and is willing to repay the loan when he can. However, he doesn't have any collateral to offer at the moment.

V1 on L006
Kofi, a 22-year-old, is seeking GHS 50,000 to start three businesses: a car washing business, a provision shop, and a phone import business from Dubai. He has no prior experience, but claims to be "business-minded" based on his friends' opinions. He promises to repay the loan within a year, once his businesses are successful, and offers no collateral, relying on his personal trustworthiness.

V2 on L002
Kwame Boateng, a commercial driver in Kumasi, has applied for a loan of GHS 25,000. He intends to use the funds to repair his trotro engine and settle personal debts. Mr. Boateng mentions that business has been slow, but he ex

Student Reasoning — Summarization prompts 1. What concrete problems did V1's output have that V2 fixed? Quote examples. 2. Why is "no invented details" an essential instruction in this application? What is this failure mode called in the LLM literature?

In [42]:
# Part 3.2 — Component 2: Structured extraction (JSON)

EXTRACT_PROMPT_SYSTEM = (
    "You are a data extraction assistant for a microfinance loan officer. "
    "Extract structured information from loan application letters. "
    "Return ONLY a JSON object — no explanation, no markdown fences, no extra text. "
    "The JSON must have EXACTLY these keys:\n"
    "  applicant_name (string)\n"
    "  amount_ghs (number)\n"
    "  purpose (string)\n"
    "  monthly_profit_ghs (number or null)\n"
    "  has_collateral_or_guarantor (boolean)\n"
    "  repayment_months (number or null)\n\n"
    "If a field is not stated in the letter, use null. Do not guess.\n\n"
    "Example letter:\n"
    "\"Dear Sir, I am Ama Serwaa, a hairdresser in Tema. I need GHS 5,000 to buy new "
    "dryers and chairs for my salon. I have no formal collateral but my husband can "
    "vouch for me informally. I hope to repay within a year.\"\n\n"
    "Example JSON output:\n"
    "{\n"
    '  "applicant_name": "Ama Serwaa",\n'
    '  "amount_ghs": 5000,\n'
    '  "purpose": "buy new dryers and chairs for salon",\n'
    '  "monthly_profit_ghs": null,\n'
    '  "has_collateral_or_guarantor": false,\n'
    '  "repayment_months": 12\n'
    "}"
)

def extract_prompt_user(letter_text):
    return f"Extract the fields from this loan application letter:\n\n{letter_text}"


import json

def extract_fields(letter_text):
    user_prompt   = extract_prompt_user(letter_text)
    answer, usage = ask_llm(user_prompt, system_prompt=EXTRACT_PROMPT_SYSTEM, temperature=0)


    cleaned  = answer.strip()
    if cleaned.startswith("```"):
        cleaned = cleaned.split("```")[1]
        if cleaned.startswith("json"):
            cleaned = cleaned[4:]
        cleaned = cleaned.strip()

    try:
        return json.loads(cleaned)
    except json.JSONDecodeError as e:
        print(f"Warning: failed to parse JSON - {e}")
        print(f"Raw output: {answer}")
        return None

import pandas as pd

rows = []
for letter_id, letter_text in LETTERS.items():
    result = extract_fields(letter_text)
    if result is not None:
        result["letter_id"] = letter_id
        rows.append(result)
    else:
        rows.append({"letter_id": letter_id})

df = pd.DataFrame(rows)
df = df[["letter_id", "applicant_name", "amount_ghs", "purpose", "monthly_profit_ghs", "has_collateral_or_guarantor", "repayment_months"]]

df




,letter_id,applicant_name,amount_ghs,purpose,monthly_profit_ghs,has_collateral_or_guarantor,repayment_months
0,L001,Akosua Mensah,8000,buy a deep freezer and expand into frozen foods,900.0,True,20.0
1,L002,Kwame Boateng,25000,repair trotro engine and settle personal debts,NaN,False,NaN
2,L003,Efua Darko,15000,purchase two industrial sewing machines and fa...,2800.0,True,15.0
3,L004,Yaw Owusu,12000,for feed and 500 new layers for poultry farm,1500.0,True,18.0
4,L005,Adenta Women's Weaving Cooperative,30000,buy a bulk order of yarn directly from the fac...,NaN,True,16.0
5,L006,Kofi,50000,"start a car washing business, a provision shop...",NaN,False,12.0


Student Reasoning — Structured extraction 1. Why must the few-shot example NOT come from the six letters you are processing? 2. Why "use null, do not guess" — what did the model do without that instruction? 3. Why is temperature=0 the right choice for extraction but arguably not for creative tasks?

In [43]:
df.columns

Index(['letter_id', 'applicant_name', 'amount_ghs', 'purpose',
       'monthly_profit_ghs', 'has_collateral_or_guarantor',
       'repayment_months'],
      dtype='str')

In [44]:
# Part 3.3 — Component 3: The decision-support brief

# TODO: Write BRIEF_PROMPT — it receives the letter AND your extracted JSON, and must output:

BRIEF_PROMPT_SYSTEM = (
    "You are a decision-support assistant for a microfinance loan officer in Ghana. "
    "Your job is to prepare a structured brief that helps the officer think through "
    "an application — you do NOT make lending decisions. Final decisions are made by "
    "human loan officers, never by you.\n\n"
    "Given a loan application letter and its extracted structured data, produce a brief "
    "with exactly these four sections:\n"
    "1. Strengths (bullet points, grounded only in what the letter actually states)\n"
    "2. Risks / red flags (bullet points)\n"
    "3. Missing information the officer should request\n"
    "4. Suggested next step — this must be a PROCESS action such as 'invite for interview', "
    "'request documents', or 'flag for senior review'. NEVER say 'approve' or 'reject', "
    "and do not imply a lending verdict anywhere in the brief."
)

def brief_prompt_user(letter_text, extracted_json):
    json_str = json.dumps(extracted_json, indent=2)
    return (
        f"Loan application letter:\n{letter_text}\n\n"
        f"Extracted data:\n{json_str}\n\n"
        f"Produce the four-section brief."
    )

briefs = {}
for letter_id, letter_text in LETTERS.items():
    extracted = extract_fields(letter_text)
    user_prompt = brief_prompt_user(letter_text, extracted)
    answer, usage = ask_llm(user_prompt, system_prompt=BRIEF_PROMPT_SYSTEM, temperature=0)
    briefs[letter_id] = answer


for letter_id in ["L001", "L002", "L006"]:
    print(f"Brief for {letter_id}")
    print(briefs[letter_id])
    print()
  

Brief for L001
## Step 1: Strengths
The applicant has several strengths that support her loan application:
* 12 years of experience selling provisions at Makola Market, indicating a stable and established business.
* A steady monthly profit of GHS 900, demonstrating a consistent income stream.
* A proven track record of saving with the susu scheme, having saved GHS 2,500 over two years without missing a contribution, which shows discipline and ability to manage finances.
* A guarantor, her sister, who is a teacher, potentially providing an additional layer of security for the loan.

## Step 2: Risks / red flags
Some potential risks and red flags associated with this application include:
* The loan amount of GHS 8,000 is significant compared to the applicant's monthly profit, which might pose a repayment risk if the business does not generate enough income.
* Expanding into frozen foods with a deep freezer could introduce new operational risks and costs, such as electricity and maintena

Student Reasoning — Decision support 1. Compare the briefs for L003 (strong application) and L006 (weak application). Did the system identify the right strengths and red flags in each? 2. Why did we forbid the model from outputting "approve"/"reject"? Give one practical and one ethical reason.

In [45]:
# Part 3.4 — Commit your prompt templates

# Commit hash: [git log -1 --format="%H"]

Section 4 — Evaluation: Quality, Reliability, Appropriateness

In [46]:
# Part 4.1 — Extraction accuracy against gold labels

GOLD_IDS = ["L001", "L003", "L006"]
FIELDS = ["applicant_name", "amount_ghs", "purpose", "monthly_profit_ghs", "has_collateral_or_guarantor", "repayment_months"]

def values_match(field, extracted_val, gold_val):
    if extracted_val is None and gold_val is None:
        return True
    if extracted_val is None or gold_val is None:
        return False
    if field == "applicant_name":
        return str(extracted_val).strip().lower() == str(gold_val).strip().lower()
    if field == "purpose":
        return str(extracted_val).strip().lower() == str(gold_val).strip().lower()
    if field == "has_collateral_or_guarantor":
        return bool(extracted_val) == bool(gold_val)
    return extracted_val == gold_val

results = {field: {} for field in FIELDS}

for letter_id in GOLD_IDS:
    extracted_row = df[df["letter_id"] == letter_id].iloc[0].to_dict()
    gold_row = GOLD[letter_id]
    for field in FIELDS:
        results[field][letter_id] = values_match(field, extracted_row.get(field), gold_row.get(field))

eval_df = pd.DataFrame(results).T
eval_df.columns = GOLD_IDS
eval_df["accuracy"] = eval_df[GOLD_IDS].mean(axis=1)

eval_df

,L001,L003,L006,accuracy
applicant_name,True,True,True,1.000000
amount_ghs,True,True,True,1.000000
purpose,False,False,False,0.000000
monthly_profit_ghs,True,True,False,0.666667
has_collateral_or_guarantor,True,True,True,1.000000
repayment_months,True,True,True,1.000000
